In [1]:
import re
from urllib.parse import urlparse
import pandas as pd
import math
from collections import Counter
from urllib.parse import urlparse

#### Address Bar Based Features

In [2]:
def url_length(url):
    return len(url)

def has_at(url):
    if '@' in url:
        return 1
    return 0

def has_dash(url):
    return url.count('-')

def dot(url):
    return url.count('.')

def no_of_subdomains(url):
    hostname = urlparse(url).netloc
    return hostname.count('.')

def has_ip(url):
    parsed = urlparse(url)
    hostname = parsed.netloc
    if re.search(r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}',hostname):
        return 1
    return 0

def shortening_service(url):
    shortners = pd.read_csv('../data/datasets/shortners.csv')
    return 1 if any(s in url for s in shortners) else 0

def double_slash_redirect(url):
    return url.count('//') if url.count('//') > 1 else 0

def ssl_final_state(url):
    return int(not url.startswith("https://"))

def https_token(url):
    hostname = urlparse(url).netloc.lower()
    return int("https" in hostname)

def redirect_symbol(url):
    return int('//' in url[8:])

def suspicious_words(url):
    keywords = ['login', 'verify', 'update', 'secure', 'account', 'banking', 'support', 'password', 'confirm']
    url_lower = url.lower()
    return sum(url_lower.count(word) for word in keywords)

def count_digits(url):
    return sum(c.isdigit() for c in url)

def path_length(url):
    parsed = urlparse(url)
    return len(parsed.path)

def suspicious_tld(url):
    bad_tlds = ['.xyz', '.top', '.tk', '.ml', '.ga', '.cf', '.gq', '.club', '.site', '.online', '.cc', '.su']
    parsed_domain = urlparse(url).netloc.lower()
    if ':' in parsed_domain:
        parsed_domain = parsed_domain.split(':')[0]
    return 1 if any(parsed_domain.endswith(tld) for tld in bad_tlds) else 0

def calculate_entropy(url):
    entropy = 0
    for x in Counter(url).values():
        px = float(x)/len(url)
        entropy -= px * math.log(px,2)
    return entropy

def domain_length(url):
    return len(urlparse(url).netloc)

def count_parameters(url):
    query = urlparse(url).query
    return len(query.split('&')) if query else 0

def has_punycode(url):
    return 1 if 'xn--' in url.lower() else 0

def count_special_chars(url):
    special_chars = ['?', '=', '_', '~', '%']
    return sum(url.count(c) for c in special_chars)

In [3]:
# 1. Brand impersonation — brand name in URL but domain isn't that brand
BRANDS = ['paypal','amazon','google','facebook','apple','microsoft','netflix',
          'instagram','twitter','linkedin','ebay','bank','wellsfargo','chase',
          'citibank','dropbox','adobe','yahoo','outlook','office365','steam',
          'youtube','whatsapp','tiktok','snapchat']
 
def brand_in_url(url):
    url_lower = url.lower()
    return sum(url_lower.count(b) for b in BRANDS)
 
def brand_domain_mismatch(url):
    """Brand name in URL but main domain is NOT that brand — strong phishing signal."""
    parsed = urlparse(url)
    hostname = parsed.netloc.lower().split(':')[0]
    # Get root domain (last 2 parts)
    parts = hostname.split('.')
    root = '.'.join(parts[-2:]) if len(parts) >= 2 else hostname
    url_lower = url.lower()
    for brand in BRANDS:
        if brand in url_lower and brand not in root:
            return 1
    return 0
 
# 2. Subdomain depth — how many subdomains (dots in hostname)
def subdomain_count(url):
    hostname = urlparse(url).netloc.lower().split(':')[0]
    parts = hostname.split('.')
    return max(0, len(parts) - 2)
 
# 3. Path depth — how many directory levels
def path_depth(url):
    path = urlparse(url).path
    return path.count('/')
 
# 4. Digit ratio — proportion of digits in full URL
def digit_ratio(url):
    if not url: return 0
    return sum(c.isdigit() for c in url) / len(url)
 
# 5. Domain digit count — digits specifically in the domain name
def domain_digit_count(url):
    hostname = urlparse(url).netloc.lower().split(':')[0]
    return sum(c.isdigit() for c in hostname)
 
# 6. Has port — non-standard port in URL (e.g. :8080)
def has_port(url):
    netloc = urlparse(url).netloc
    return int(bool(re.search(r':\d+$', netloc)))
 
# 7. TLD legitimacy score — trusted TLDs get 0 (legit), unknown get 1
def tld_legitimacy(url):
    trusted = ['.com','.org','.net','.edu','.gov','.co','.io','.uk',
               '.de','.fr','.jp','.au','.ca','.in','.it','.es','.nl']
    host = urlparse(url).netloc.lower().split(':')[0]
    return 0 if any(host.endswith(t) for t in trusted) else 1
 
# 8. Longest URL token — max length of any segment split by /.-_?=&
def longest_token(url):
    tokens = re.split(r'[/.\-_?=&#]', url)
    return max((len(t) for t in tokens), default=0)
 
# 9. Vowel ratio in domain — random/gibberish domains have unusual vowel ratios
def vowel_ratio_domain(url):
    hostname = urlparse(url).netloc.lower().split(':')[0]
    alpha = [c for c in hostname if c.isalpha()]
    if not alpha: return 0
    vowels = sum(c in 'aeiou' for c in alpha)
    return vowels / len(alpha)
 
# 10. Hex encoded characters count — %20 %3D etc (obfuscation)
def hex_char_count(url):
    return len(re.findall(r'%[0-9a-fA-F]{2}', url))
 
# 11. Consecutive repeated characters — aaaa or 1111 patterns (random generation)
def max_consecutive_repeated(url):
    if not url: return 0
    max_rep = 1
    cur = 1
    for i in range(1, len(url)):
        if url[i] == url[i-1]:
            cur += 1
            max_rep = max(max_rep, cur)
        else:
            cur = 1
    return max_rep
 
# 12. Has fragment — # in URL (rarely used legitimately in phishing)
def has_fragment(url):
    return int(bool(urlparse(url).fragment))
 
# 13. Unique char ratio — low = repetitive/random URL
def unique_char_ratio(url):
    if not url: return 0
    return len(set(url.lower())) / len(url)
 
# 14. Domain entropy — entropy specifically of the domain name
def domain_entropy(url):
    hostname = urlparse(url).netloc.lower().split(':')[0]
    if not hostname: return 0
    cnt = Counter(hostname)
    return -sum((v/len(hostname))*math.log(v/len(hostname),2) for v in cnt.values())
 
# 15. Has www — legitimate sites more often use www
def has_www(url):
    return int(urlparse(url).netloc.lower().startswith('www.'))
 
# ── Combined extractor ────────────────────────────────────────────────────────
def extract_all_features_v3(url, shorteners=None):
    return pd.Series({
        'url_length':           url_length(url),
        'has_at':               has_at(url),
        'has_dash':             has_dash(url),
        'dot':                  dot(url),
        'has_ip':               has_ip(url),
        'shortening_service':   shortening_service(url),
        'double_slash_redirect':double_slash_redirect(url),
        'ssl_final_state':      ssl_final_state(url),
        'https_token':          https_token(url),
        'redirect_symbol':      redirect_symbol(url),
        'suspicious_words':     suspicious_words(url),
        'digits':               count_digits(url),
        'path_length':          path_length(url),
        'suspicious_tld':       suspicious_tld(url),
        'entropy':              calculate_entropy(url),
        'domain_length':        domain_length(url),
        'paramaters':           count_parameters(url),
        'has_punycode':         has_punycode(url),
        'special_chars':        count_special_chars(url),
        'brand_in_url':         brand_in_url(url),
        'brand_domain_mismatch':brand_domain_mismatch(url),
        'subdomain_count':      subdomain_count(url),
        'path_depth':           path_depth(url),
        'digit_ratio':          digit_ratio(url),
        'domain_digit_count':   domain_digit_count(url),
        'has_port':             has_port(url),
        'tld_legitimacy':       tld_legitimacy(url),
        'longest_token':        longest_token(url),
        'vowel_ratio_domain':   vowel_ratio_domain(url),
        'hex_char_count':       hex_char_count(url),
        'max_consecutive_rep':  max_consecutive_repeated(url),
        'has_fragment':         has_fragment(url),
        'unique_char_ratio':    unique_char_ratio(url),
        'domain_entropy':       domain_entropy(url),
        'has_www':              has_www(url),
    })

In [5]:
data = pd.read_csv('../data/datasets/raw_dataset.csv')
columns = data['url'].apply(extract_all_features_v3)
columns['label'] = data['label']

columns.to_csv('../data/datasets/processed_dataset_v3.csv',index=False)
print("Raw data is processed successfully...")

Raw data is processed successfully...
